In [1]:
import datasets

/home/joregan/miniconda3/envs/hfnew/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
vctk = datasets.load_dataset("kth-tmh/vctk", "default")

In [3]:
if "path" in vctk["train"][0]["audio"]:
    vctk = vctk.filter(lambda example: example["audio"]['path'].endswith("_mic2.flac"))
else:
    vctk = vctk.filter(lambda example: example["audio"].__dict__['_hf_encoded']['path'].endswith("_mic2.flac"))

In [4]:
from datasets import Audio
vctk = vctk.cast_column("audio", Audio(sampling_rate=16000))

In [5]:
gbi = datasets.load_dataset("kth-tmh/google-britain-ireland")

In [6]:
gbi = gbi.cast_column("audio", Audio(sampling_rate=16000))

In [7]:
dialects = set()
for dialect in gbi['train']['dialect']:
    dialects.add(dialect)
print(dialects)

{'Irish', 'Scottish', 'Welsh', 'Northern English', 'Midlands English', 'Southern English'}


In [8]:
def remap_gbi_sentence_id(item):
    if item['sentence_id'] == 'EN0628':
        item['sentence_id'] = 'EN0470'
    return item
gbi.map(remap_gbi_sentence_id)

DatasetDict({
    train: Dataset({
        features: ['sentence_id', 'utterance_id', 'text', 'speaker_id', 'dialect', 'gender', 'audio'],
        num_rows: 17877
    })
})

In [9]:
# vctk: delete these speaker IDs (no mic2)
NO_MIC2 = ['p315', 'p280']
vctk = vctk.filter(lambda example: example['speaker_id'] not in NO_MIC2)

In [10]:
import json

with open("utterance-map.json") as f:
    utterance_map = json.load(f)

In [11]:
def remap_vctk(item):
    if item['region'] in ['Southern  England', 'Surrey', 'Essex', 'London', 'Oxford', 'Suffolk', 'SW  England', 'SE  England']:
        item['dialect'] = 'Southern English'
    if item['region'] in ['Birmingham', 'Leicester', 'Nottingham', 'Staffordshire']:
        item['dialect'] = 'Midlands English'
    if item['region'] in ['Manchester', 'Cumbria', 'Stockton-on-tees', 'Cheshire', 'Newcastle', 'York', 'Yorkshire']:
        item['dialect'] = 'Northern English'
    if item['accent'] == 'NorthernIrish':
        item['dialect'] = 'Irish'
    if item['accent'] == 'SouthAfrican':
        item['dialect'] = 'South African'
    if item['accent'] == 'NewZealand':
        item['dialect'] = 'New Zealand'
    if item['accent'] in ['Scottish', 'Irish', 'American', 'Canadian', 'Australian']:
        item['dialect'] = item['accent']

    if not 'dialect' in item:
        item['dialect'] = None
    del item['region']
    del item['accent']
    del item['age']

    sentence_key = f"{item['speaker_id']}_{item['text_id']}"

    if sentence_key in utterance_map["vctk"]:
        item['sentence_id'] = utterance_map["vctk"][sentence_key]
    else:
        item['sentence_id'] = sentence_key
    item['utterance_id'] = sentence_key

    del item['text_id']

    return item

In [12]:
vctk = vctk.map(remap_vctk)

In [13]:
# filter to remove elements where 'dialect' == None
vctk = vctk.filter(lambda example: example['dialect'] is not None)

In [14]:
cmu_arctic = datasets.load_dataset("kth-tmh/cmu_arctic")

In [15]:
cmu_arctic = cmu_arctic.cast_column("audio", Audio(sampling_rate=16000))

In [16]:
cmu_arctic

DatasetDict({
    train: Dataset({
        features: ['audio', 'text', 'sentence_id', 'speaker_id', 'gender', 'accent'],
        num_rows: 15583
    })
})

In [17]:
SIMILAR = {
    "jmk": {
        "arctic_b0028": "arctic_b0028"
    },
    "awb": {
        "arctic_a0335": "arctic_a0333",
        "arctic_a0336": "arctic_a0334",
        "arctic_a0433": "arctic_c0036",
        "arctic_a0557": "arctic_a0554",
        "arctic_b0211": "arctic_b0211",
        "arctic_b0234": "arctic_b0234"
    }
}

In [18]:
utterance_map.keys()

dict_keys(['vctk', 'awb', 'awb_different_norm', 'gbi'])

Missing: arctic_b0028 jmk lord fitzhugh is the key to this whole situation
Missing: arctic_a0335 awb this is no place fer you
Missing: arctic_a0336 awb he'll knock yeh off a few sticks in no time
Missing: arctic_a0433 awb her true course had been west 0.75 south
Missing: arctic_a0557 awb jack london waikiki beach honolulu oahu t.h
Missing: arctic_a0567 awb an hallucination began to trouble him
Missing: arctic_b0211 awb this is 1880
Missing: arctic_b0234 awb and watch out for wet feet,' was his parting advice
Missing: arctic_b0419 awb to begin with i read late
Missing: arctic_b0426 awb there were orange green gold green and a copper green
Missing: arctic_b0445 awb he sat beside the gendarme and beamed

In [19]:
vctk["train"][0]

{'speaker_id': 'p226',
 'audio': {'path': 'p226_001_mic2.flac',
  'array': array([-0.00977735, -0.01614734, -0.01399516, ..., -0.07560794,
         -0.08644344,  0.        ]),
  'sampling_rate': 16000},
 'file': '/home/joregan/.cache/huggingface/datasets/downloads/extracted/c530a57fbfe03b9c72fce050387427725c03a69fac40d8ea1913387ec2ed2f4f/wav48_silence_trimmed/p226/p226_001_mic2.flac',
 'text': 'Please call Stella.',
 'gender': 'M',
 'comment': '',
 'dialect': 'Southern English',
 'sentence_id': 's5_001',
 'utterance_id': 'p226_001'}

In [ ]:
MISSED_NORMS = {
    "arctic_a0343": "arctic_a0341",
    "arctic_a0427": "arctic_a0425",
    "arctic_b0453": "arctic_b0451",
    "arctic_b0030": "arctic_b0030",
    "arctic_b0365": "arctic_b0365",
    "arctic_b0348": "arctic_b0348",
    "arctic_a0181": "arctic_a0179",
    "arctic_a0063": "arctic_a0063",
    "arctic_b0208": "arctic_b0208",
    "arctic_b0434": "arctic_b0433",
    "arctic_a0301": "arctic_a0299",
    "arctic_a0091": "arctic_a0089",
    "arctic_a0260": "arctic_a0258",
    "arctic_a0060": "arctic_a0060",
    "arctic_b0313": "arctic_b0313",
    "arctic_b0493": "arctic_b0491",
    "arctic_a0167": "arctic_a0165",
    "arctic_a0249": "arctic_a0247",
    "arctic_a0294": "arctic_a0292",
    "arctic_a0260": "arctic_a0258",
    "arctic_a0592": "arctic_a0588",
    "arctic_b0377": "arctic_b0377",
    "arctic_b0199": "arctic_b0199",
    "arctic_b0280": "arctic_b0280",
    "arctic_a0461": "arctic_a0458",
    "arctic_a0543": "arctic_a0540",
    "arctic_a0392": "arctic_a0390",
    "arctic_b0374": "arctic_b0374",
    "arctic_a0152": "arctic_a0150",
    "arctic_b0107": "arctic_b0107",
    "arctic_b0204": "arctic_b0204",
    "arctic_b0080": "arctic_b0080",
    "arctic_b0334": "arctic_b0334",
    "arctic_a0094": "arctic_a0092",
    "arctic_b0166": "arctic_b0166",
    "arctic_b0041": "arctic_b0041",
    "arctic_a0103": "arctic_a0101",
    "arctic_b0228": "arctic_b0228",
    "arctic_a0001": "arctic_a0001",
    "arctic_b0248": "arctic_b0248",
    "arctic_b0132": "arctic_b0132",
    "arctic_b0181": "arctic_b0181",
    "arctic_b0192": "arctic_b0192",
    "arctic_b0173": "arctic_b0173",
    "arctic_b0274": "arctic_b0274",
    "arctic_b0376": "arctic_b0376",
    "arctic_b0167": "arctic_b0167",
    "arctic_b0013": "arctic_b0013",
    "arctic_b0330": "arctic_b0330",
    "arctic_b0213": "arctic_b0213",
    "arctic_b0344": "arctic_b0344",
    "arctic_b0307": "arctic_b0307",
    "arctic_b0171": "arctic_b0171",
    "arctic_b0091": "arctic_b0091",
    "arctic_b0516": "arctic_b0514",
    "arctic_b0144": "arctic_b0144",
}

In [21]:
todo = []

In [ ]:
def remap_cmu_arctic(item):
    item['utterance_id'] = f"cmu_us_{item['speaker_id']}_{item['sentence_id']}"
    if item['sentence_id'] == "arctic_c0035":
        item['sentence_id'] = "arctic_a0456"

    if item['speaker_id'] == "jmk" and item['sentence_id'] == "arctic_b0028":
        item['sentence_id'] = "jmk_arctic_b0028"
    
    unique = ["arctic_b0445", "arctic_b0426", "arctic_b0419", "arctic_a0567", "arctic_a0074"]
    diff_enough = ["arctic_a0335", "arctic_a0557", "arctic_a0336"]

    if item['speaker_id'] == "awb":
        if item['sentence_id'] in utterance_map["awb"]:
            item['sentence_id'] = utterance_map["awb"][item['sentence_id']]
        elif item['sentence_id'] in utterance_map["awb_different_norm"]:
            item['sentence_id'] = utterance_map["awb_different_norm"][item['sentence_id']]
        elif item['sentence_id'] in MISSED_NORMS:
            item['sentence_id'] = MISSED_NORMS[item['sentence_id']]
        elif item['sentence_id'] in unique:
            item['sentence_id'] = f"awb_{item['sentence_id']}"
        elif item['sentence_id'] in diff_enough:
            item['sentence_id'] = f"awb_{item['sentence_id']}"
        elif item['sentence_id'] in ["arctic_b0234", "arctic_a0433", "arctic_b0211"]:
            item['sentence_id'] = item['sentence_id']
        else:
            # raise ValueError(f"Unexpected sentence_id {item['sentence_id']} for speaker awb")
            todo.append(item['sentence_id'])
            print(f"Unexpected sentence_id {item['sentence_id']} for speaker awb")

    return item

In [23]:
cmu_arctic = cmu_arctic.map(remap_cmu_arctic)

Map:  37%|███▋      | 5690/15583 [00:01<00:03, 3037.00 examples/s]

Unexpected sentence_id arctic_b0041 for speaker awb
Unexpected sentence_id arctic_a0103 for speaker awb
Unexpected sentence_id arctic_b0228 for speaker awb
Unexpected sentence_id arctic_a0001 for speaker awb
Unexpected sentence_id arctic_b0248 for speaker awb


Map:  41%|████▏     | 6461/15583 [00:01<00:03, 2943.54 examples/s]

Unexpected sentence_id arctic_b0132 for speaker awb
Unexpected sentence_id arctic_b0181 for speaker awb
Unexpected sentence_id arctic_b0192 for speaker awb
Unexpected sentence_id arctic_b0173 for speaker awb
Unexpected sentence_id arctic_b0274 for speaker awb
Unexpected sentence_id arctic_b0376 for speaker awb
Unexpected sentence_id arctic_b0167 for speaker awb
Unexpected sentence_id arctic_b0013 for speaker awb
Unexpected sentence_id arctic_a0074 for speaker awb
Unexpected sentence_id arctic_b0330 for speaker awb
Unexpected sentence_id arctic_b0213 for speaker awb
Unexpected sentence_id arctic_b0344 for speaker awb
Unexpected sentence_id arctic_b0307 for speaker awb
Unexpected sentence_id arctic_b0171 for speaker awb
Unexpected sentence_id arctic_b0091 for speaker awb
Unexpected sentence_id arctic_b0516 for speaker awb
Unexpected sentence_id arctic_b0144 for speaker awb


Map: 100%|██████████| 15583/15583 [00:07<00:00, 2174.13 examples/s]


In [27]:
[x.replace("awb_", "") for x in todo]

['arctic_b0041',
 'arctic_a0103',
 'arctic_b0228',
 'arctic_a0001',
 'arctic_b0248',
 'arctic_b0132',
 'arctic_b0181',
 'arctic_b0192',
 'arctic_b0173',
 'arctic_b0274',
 'arctic_b0376',
 'arctic_b0167',
 'arctic_b0013',
 'arctic_a0074',
 'arctic_b0330',
 'arctic_b0213',
 'arctic_b0344',
 'arctic_b0307',
 'arctic_b0171',
 'arctic_b0091',
 'arctic_b0516',
 'arctic_b0144']